# Mixture of Experts (MoE)

A refresher on the architecture behind today's largest open LLMs (Mixtral, DeepSeek-V3,
Qwen-MoE, Switch Transformer) — **scale parameter count without scaling compute** by
activating only a sparse subset of the network per token.

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

A Mixture of Experts replaces one big dense layer (usually the feed-forward block of a
Transformer) with **many parallel "expert" sub-networks plus a small router**. For each
input token the router picks the top-*k* experts (typically *k*=1 or 2 out of 8–256), runs
*only those*, and combines their outputs by the router's weights. The other experts sit idle.

**The problem it solves.** Model quality scales with parameter count, but a *dense* model
pays FLOPs proportional to *every* parameter on *every* token. That gets ruinously
expensive. MoE decouples the two: you can grow total parameters 10–50× (more knowledge,
more capacity) while keeping the **per-token compute roughly fixed**, because each token
only ever touches a handful of experts. Mixtral-8x7B has ~47B parameters but the FLOPs of a
~13B dense model; DeepSeek-V3 has 671B parameters with only ~37B active per token.

**Reach for it when** you are training/serving at scale and are *memory-bandwidth and
parameter rich but FLOP constrained* — you want a bigger model than your compute budget
allows densely. **Avoid it when** you can't hold all experts in memory (every expert must be
resident even though most are idle), when your batch is tiny (routing overhead and load
imbalance dominate), or when a dense model already fits your quality target — MoE adds real
systems complexity. See [Vision Transformer](vision-transformer.ipynb) and
[Mamba / SSM](mamba-ssm.ipynb) for the dense backbones MoE layers usually slot into.

## 2. Mental Model

**A hospital with specialists and a triage nurse.**

A dense FFN is one exhausted generalist doctor who sees *every* patient and must know
everything. An MoE clinic instead has a roster of specialists (the **experts**) and a fast
**triage nurse** (the **router/gate**) at the door. Each patient (token) is glanced at,
routed to the 1–2 most relevant specialists, treated, and the opinions are blended by how
confident the nurse was. The clinic can hire 100 specialists — vast collective knowledge —
yet each patient only ever occupies two of them, so throughput stays high.

```
                  ┌──────────────────────────────┐
   token x ─────▶ │ router  (small Linear → E)   │
                  └──────────────┬───────────────┘
                       softmax over top-k logits
              ┌────────────┬─────┴──────┬───────────────┐
              ▼            ▼            (idle)          (idle)
          Expert 3     Expert 7      Expert 1   ...   Expert E
           (FFN)        (FFN)
              └──── w3·· ┘  └ ··w7 ────┘
                          ▼
                    y = w3·E3(x) + w7·E7(x)         # k=2 here
```

The whole trick: the router is *cheap* (one small matmul) and *sparse* (it commits to a
few experts), so adding experts adds parameters and knowledge but **not** per-token FLOPs.

## 3. Key Concepts

- **Expert** — an independent sub-network, almost always a standard Transformer FFN
  (`Linear → activation → Linear`). A layer holds `E` of them (8, 64, 256…).
- **Router / gate** — a tiny `Linear(d_model → E)` producing a logit per expert. `softmax`
  over the chosen experts gives the combination weights. This is the *only* part that learns
  *which* expert handles *what*.
- **Top-k routing** — each token is sent to its `k` highest-scoring experts (`k`=1 in Switch
  Transformer, `k`=2 in Mixtral/GShard). `k` controls the compute/quality trade-off.
- **Sparse vs dense activation** — "sparse MoE" = only `k` of `E` experts run per token. A
  *soft*/dense MoE runs all experts and weights them (no FLOP savings; rare at scale).
- **Active vs total parameters** — total = all experts; active = the `~k/E` fraction touched
  per token. Quality tracks *total*; cost tracks *active*. This gap is the entire point.
- **Load balancing** — left alone, the router collapses onto a few favorite experts (rich
  get richer). An **auxiliary load-balancing loss** (Switch Transformer) pushes tokens to
  spread evenly across experts so capacity isn't wasted.
- **Capacity factor & token dropping** — for fixed-shape batched compute each expert has a
  buffer of `capacity = capacity_factor · tokens · k / E`. Tokens over the buffer are
  *dropped* (skip the layer via the residual). Higher capacity = fewer drops but more waste.
- **Expert parallelism** — experts are sharded across devices; routing becomes an
  all-to-all communication of tokens to the device holding their expert. The dominant
  systems cost of MoE.
- **Shared experts** — newer designs (DeepSeek-MoE) keep 1–2 *always-on* shared experts for
  common patterns plus many routed experts for specialization.

## 4. Setup

Pure CPU PyTorch — no GPU, no downloads. The worked examples build and train a tiny MoE
layer on random tensors in a couple of seconds.

```bash
%pip install torch numpy
# Optional, only for the gated real-model demo at the end:
%pip install transformers accelerate
```

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print("torch", torch.__version__, "| running on CPU")

torch 2.12.1 | running on CPU


## 5. Worked Examples

### Example 1 — A sparse top-2 MoE layer, end to end

Build `E=8` expert FFNs plus a router, route a batch of tokens to their **top-2** experts,
and combine the outputs by the (renormalized) router weights. The two numbers that matter:
the **active-parameter fraction** (why MoE is cheap) and the **per-expert token load** (which
motivates Example 2).

In [2]:
class Expert(nn.Module):
    """A standard Transformer FFN — the unit an MoE replicates."""
    def __init__(self, d, h):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Linear(h, d))
    def forward(self, x):
        return self.net(x)

class SparseMoE(nn.Module):
    def __init__(self, d=32, h=64, n_experts=8, k=2):
        super().__init__()
        self.k, self.n_experts = k, n_experts
        self.experts = nn.ModuleList([Expert(d, h) for _ in range(n_experts)])
        self.router = nn.Linear(d, n_experts)          # the cheap gate

    def forward(self, x):                              # x: (tokens, d)
        logits = self.router(x)                        # (T, E)  one score per expert
        weights, idx = torch.topk(logits, self.k, dim=-1)   # keep top-k per token
        weights = F.softmax(weights, dim=-1)           # normalize over the chosen k
        out = torch.zeros_like(x)
        load = torch.zeros(self.n_experts)
        for slot in range(self.k):                     # gather tokens per expert, run once
            for e in range(self.n_experts):
                m = idx[:, slot] == e
                if m.any():
                    out[m] += weights[m, slot:slot+1] * self.experts[e](x[m])
                    load[e] += int(m.sum())
        return out, logits, idx, load

T, d, E, k = 512, 32, 8, 2
moe = SparseMoE(d=d, h=64, n_experts=E, k=k)
x = torch.randn(T, d)
out, logits, idx, load = moe(x)

total  = sum(p.numel() for p in moe.parameters())
active = (sum(p.numel() for p in moe.experts[0].parameters()) * k
          + sum(p.numel() for p in moe.router.parameters()))
print("output shape:", tuple(out.shape))
print(f"params: total={total:,}  active/token≈{active:,}  ({100*active/total:.0f}% of the model runs)")
print("per-expert token load:", load.int().tolist(), " sum =", int(load.sum()), "(= T*k)")

output shape: (512, 32)
params: total=33,800  active/token≈8,648  (26% of the model runs)
per-expert token load: [135, 97, 112, 146, 180, 83, 172, 99]  sum = 1024 (= T*k)


The output has the same shape as a dense FFN would produce, but only ~26% of the
parameters ran per token. Note the **load is already lopsided** (e.g. one expert sees ~180
tokens, another ~83) even with random data and an untrained router — that imbalance is the
central failure mode MoE training has to fight.

### Example 2 — The load-balancing auxiliary loss

Without intervention the router collapses onto a few favored experts: they get more
gradient, get better, get chosen even more. The fix is Switch Transformer's
**load-balancing loss** — `E/k · Σ (fraction of tokens dispatched to expert e) · (mean
router probability for e)` — which is minimized exactly when tokens spread *uniformly*.
We train **only the router** on it and watch the load even out, measured by the coefficient
of variation (CV) of per-expert load (0 = perfectly balanced).

In [3]:
def aux_loss(logits, idx, E, k):
    """Switch-Transformer load-balancing loss (minimized at a uniform split)."""
    P = F.softmax(logits, dim=-1).mean(0)                    # mean router prob per expert
    f = F.one_hot(idx.reshape(-1), E).float().mean(0)        # dispatch fraction per expert
    return E / k * torch.sum(f * P)

def cv(load):                                                # 0.0 == perfectly balanced
    return (load.std() / load.mean()).item()

_, logits0, idx0, load0 = moe(x)
print(f"before balancing: load CV = {cv(load0):.3f}   {load0.int().tolist()}")

opt = torch.optim.Adam(moe.router.parameters(), lr=0.05)    # train ONLY the router
for step in range(200):
    _, logits, idx, _ = moe(x)
    loss = aux_loss(logits, idx, E, k)
    opt.zero_grad(); loss.backward(); opt.step()

_, _, _, load1 = moe(x)
print(f"after  balancing: load CV = {cv(load1):.3f}   {load1.int().tolist()}")

before balancing: load CV = 0.282   [135, 97, 112, 146, 180, 83, 172, 99]


after  balancing: load CV = 0.020   [128, 123, 130, 128, 132, 128, 128, 127]


The CV collapses (≈0.28 → ≈0.02) and every expert ends up handling ~`T·k/E` tokens. In a
real model this auxiliary loss is added to the main language-modeling loss with a small
coefficient (≈0.01); too large and it fights task learning, too small and experts collapse.

### Example 3 — A real pretrained MoE (gated)

The toy layer above *is* the mechanism; production models (Mixtral-8x7B, DeepSeek-V3) wrap
the same router-plus-experts pattern in a full Transformer with expert parallelism. The call
shape below loads Mixtral via 🤗 Transformers — gated behind an env var because the weights
are a ~90 GB download, so this notebook still executes without it.

In [4]:
# Set RUN_HF=1 to actually download + run a real MoE LLM (very large, GPU strongly advised).
if os.getenv("RUN_HF"):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "mistralai/Mixtral-8x7B-v0.1"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", load_in_4bit=True)
    ids = tok("Mixture of experts works by", return_tensors="pt").to(model.device)
    print(tok.decode(model.generate(**ids, max_new_tokens=20)[0]))
else:
    print("Skipped real MoE model (set RUN_HF=1 to enable). Call shape:")
    print('  from transformers import AutoModelForCausalLM, AutoTokenizer')
    print('  name  = "mistralai/Mixtral-8x7B-v0.1"   # 8 experts, top-2, ~47B total / ~13B active')
    print('  model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", load_in_4bit=True)')
    print('  # forward pass routes each token through 2 of 8 FFN experts per layer')

Skipped real MoE model (set RUN_HF=1 to enable). Call shape:
  from transformers import AutoModelForCausalLM, AutoTokenizer
  name  = "mistralai/Mixtral-8x7B-v0.1"   # 8 experts, top-2, ~47B total / ~13B active
  model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", load_in_4bit=True)
  # forward pass routes each token through 2 of 8 FFN experts per layer


## 6. Gotchas & Pitfalls

- **Router collapse / dead experts.** The default failure mode: a few experts win, the rest
  never get tokens and never learn. Always use a load-balancing (and often a *router-z*)
  loss; monitor per-expert load during training, not just final loss.
- **Total params ≠ active params.** Marketing says "47B"; you serve the FLOPs of ~13B but
  must hold **all 47B in memory**. MoE saves compute, *not* memory — every idle expert is
  still resident. This surprises people sizing inference hardware.
- **Token dropping under capacity limits.** Batched MoE gives each expert a fixed buffer;
  overflow tokens are silently dropped to the residual. Too-low `capacity_factor` quietly
  degrades quality; too-high wastes compute on padding. Watch the drop rate.
- **Tiny batches kill the win.** With few tokens per expert per step, the all-to-all routing
  overhead and load imbalance dominate. MoE pays off at large batch × sequence, not on a
  single short prompt.
- **Training instability.** Sparse routing makes loss spikier than dense; the top-k argmax is
  non-differentiable (gradients flow only through the chosen experts' weights). Router-z
  loss, careful init, and bf16 (not fp16) help.
- **Aux-loss coefficient is finicky.** Too high and balancing fights the task; too low and
  experts collapse. ≈0.01 is the usual starting point — tune it, don't ignore it.
- **Fine-tuning is harder.** Small downstream datasets can re-collapse routing or overfit a
  subset of experts; MoE models are more delicate to fine-tune than dense ones.
- **Reproducibility of routing.** Padding, batch composition, and even sequence packing
  change which tokens an expert sees, so outputs can shift with batch shape. Pin these when
  debugging.

## 7. When to Use vs Alternatives

| Approach | Params vs compute | Quality/$ at scale | Memory | Complexity | Notes |
|---|---|---|---|---|---|
| **Sparse MoE (top-k)** | Decoupled — huge params, small FLOPs | **Best** when FLOP-bound | High (all experts resident) | High (routing, balancing, expert parallelism) | Mixtral, DeepSeek-V3, Switch, GShard |
| **Dense FFN Transformer** | Coupled — FLOPs ∝ params | Best when memory-bound or small | Lower (no idle experts) | Low | The default; simplest to train/serve |
| **Wider/deeper dense scaling** | Coupled | Diminishing — compute explodes | Grows with params | Low | Hits a compute wall MoE sidesteps |
| **Distillation to a small dense model** | Coupled, tiny | Great for deployment | Lowest | Medium (needs a teacher) | Serve cheap after training big |
| **Soft / dense MoE** | All experts run | No FLOP saving | High | Medium | Differentiable routing; rarely worth it at scale |

**Rules of thumb.** Choose **MoE** when you are FLOP-limited and want more model than dense
compute allows, you can fit all experts in (aggregate) memory, and you run large batches —
i.e. frontier-scale pretraining and high-throughput serving. Stick with a **dense**
Transformer when the model already fits your quality target, memory (not compute) is the
binding constraint, batches are small, or you want operational simplicity. If you trained an
MoE but must deploy cheaply on constrained hardware, **distill** it into a dense student.

## 8. Resources

- **Shazeer et al. — Outrageously Large Neural Networks (Sparsely-Gated MoE, 2017)**, the
  paper that revived MoE for deep learning: https://arxiv.org/abs/1701.06538
- **Fedus, Zoph & Shazeer — Switch Transformers (2021)**, top-1 routing and the
  load-balancing loss used above: https://arxiv.org/abs/2101.03961
- **Mixtral of Experts (Jiang et al., 2024)**, the open 8x7B top-2 model:
  https://arxiv.org/abs/2401.04088
- **DeepSeek-V3 Technical Report (2024)**, shared experts + fine-grained MoE at 671B:
  https://arxiv.org/abs/2412.19437
- **Hugging Face — "Mixture of Experts Explained"**, the best practical walkthrough:
  https://huggingface.co/blog/moe
- **GShard (Lepikhin et al., 2020)**, expert parallelism and capacity factors:
  https://arxiv.org/abs/2006.16668